<a href="https://colab.research.google.com/github/MSDEEPAK013/CatsvsDogs-Classification-using-CNN/blob/main/Pruning-%20on%20mnist.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install tensorflow-model-optimization

import tensorflow as tf
import tensorflow_model_optimization as tfmot
from tensorflow.keras import layers, models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 1.8 MB/s eta 0:00:00


In [6]:
mnist = tf.keras.datasets.mnist
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [7]:
train_images = train_images.astype('float32') / 255.0
test_images = test_images.astype('float32') / 255.0
train_images = train_images[..., tf.newaxis]
test_images = test_images[..., tf.newaxis]

In [28]:

def model():
  model=models.Sequential([
      layers.Conv2D(32,(3,3),activation='relu',input_shape=(28,28,1)),
      layers.MaxPooling2D((2,2)),
      layers.Conv2D(64,(3,3),activation='relu'),
      layers.MaxPool2D((2,2)),
      layers.Conv2D(64,(3,3),activation='relu'),
      layers.Flatten(),
      layers.Dense(64,activation='relu'),
      layers.Dense(10,activation='softmax')

  ])
  return model

In [45]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"


epochs = 4
batch_size = 64
num_images = train_images.shape[0]
end_step = (num_images // batch_size) * epochs

prune_para = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.50,
        begin_step=0,
        end_step=end_step
    )
}


base_model = model()
model_for_pruning = tfmot.sparsity.keras.prune_low_magnitude(base_model, **prune_para)


In [46]:
model_for_pruning.compile(optimizer='adam',
              loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy'])

model_for_pruning.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_conv2d  (None, 26, 26, 32)        610       
 _6 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_max_po  (None, 13, 13, 32)        1         
 oling2d_4 (PruneLowMagnitu                                      
 de)                                                             
                                                                 
 prune_low_magnitude_conv2d  (None, 11, 11, 64)        36930     
 _7 (PruneLowMagnitude)                                          
                                                                 
 prune_low_magnitude_max_po  (None, 5, 5, 64)          1         
 oling2d_5 (PruneLowMagnitu                                      
 de)                                                  

In [47]:
callback = [tfmot.sparsity.keras.UpdatePruningStep()]

model_for_pruning.fit(train_images, train_labels,
                  batch_size=batch_size, epochs=epochs, validation_split=0.1,
                  callbacks=callback)

Epoch 1/4


/usr/local/lib/python3.13/dist-packages/keras/src/backend/tensorflow/nn.py:1214: UserWarning: "`sparse_categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


844/844 [==============================] - 51s 55ms/step - loss: 0.1797 - accuracy: 0.9447 - val_loss: 0.0622 - val_accuracy: 0.9833
Epoch 2/4
844/844 [==============================] - 47s 56ms/step - loss: 0.0498 - accuracy: 0.9848 - val_loss: 0.0449 - val_accuracy: 0.9878
Epoch 3/4
844/844 [==============================] - 46s 54ms/step - loss: 0.0345 - accuracy: 0.9894 - val_loss: 0.0398 - val_accuracy: 0.9890
Epoch 4/4
844/844 [==============================] - 47s 56ms/step - loss: 0.0256 - accuracy: 0.9919 - val_loss: 0.0434 - val_accuracy: 0.9873


In [48]:
_, pruned_accuracy = model_for_pruning.evaluate(test_images, test_labels, verbose=0)
print(f"\nPruned Test Accuracy: {pruned_accuracy * 100:.2f}%")



Pruned Test Accuracy: 99.03%


In [54]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tf_keras as keras

# 1. Redefine model() explicitly with SQUARE BRACKETS []
def model():
    return keras.Sequential([
        keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation='relu'),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Conv2D(64, (3, 3), activation='relu'),
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(10)  # No activation here because from_logits=True
    ])

# 2. Instantiate new model instance
baseline_model = model()

# 3. Compile
baseline_model.compile(
    optimizer='adam',
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# 4. Fit and check performance
baseline_model.fit(
    train_images,
    train_labels,
    batch_size=64,
    epochs=4,
    validation_split=0.1
)

Epoch 1/4
844/844 [==============================] - 48s 55ms/step - loss: 0.1984 - accuracy: 0.9387 - val_loss: 0.0563 - val_accuracy: 0.9838
Epoch 2/4
844/844 [==============================] - 46s 55ms/step - loss: 0.0545 - accuracy: 0.9830 - val_loss: 0.0427 - val_accuracy: 0.9885
Epoch 3/4
844/844 [==============================] - 45s 53ms/step - loss: 0.0378 - accuracy: 0.9876 - val_loss: 0.0526 - val_accuracy: 0.9860
Epoch 4/4
844/844 [==============================] - 46s 55ms/step - loss: 0.0291 - accuracy: 0.9907 - val_loss: 0.0358 - val_accuracy: 0.9902


In [57]:
_, baseline_accuracy= baseline_model.evaluate(test_images, test_labels, verbose=0)
print(f"\nBaseline Test Accuracy: {baseline_accuracy * 100:.2f}%")



Baseline Test Accuracy: 98.95%
